# EX_12 — Contexto largo y multimodal (ejercicios)

**Notebook de referencia:** `notebook/12_Modelos_Contexto_Multimodales.ipynb`

**Tiempo orientativo:** ~30 minutos.


## Actividad 1 — Presupuesto de tokens

Estima (orden de magnitud) tokens para: 10 páginas de texto, 1 imagen 1024×1024 en un modelo que la trata como patches, y 2 minutos de audio crudo 16 kHz 16-bit. Usa markdown con supuestos explícitos.


_Supuestos y estimaciones:_

...


### 📊 Actividad 1 — Presupuesto de Tokens Multimodal

A continuación, se detalla la estimación del orden de magnitud de tokens para cada tipo de modalidad de entrada, basándose en supuestos técnicos estándar de la industria.

---

#### 1. Texto: 10 páginas de texto
* **Supuestos explícitos:**
  * Una página estándar de texto (formato A4/Carta, espaciado normal) contiene aproximadamente entre **400 y 500 palabras**.
  * En español/inglés, el factor de conversión promedio de los tokenizadores modernos (como Tiktoken o Llama) es de aprox. **1 palabra ≈ 1.3 a 1.4 tokens**.
* **Cálculo:** $$\text{Total palabras} = 10 \text{ páginas} \times 500 \text{ palabras/página} = 5,000 \text{ palabras}$$
  $$\text{Total tokens} = 5,000 \times 1.4 \approx 7,000 \text{ tokens}$$
* **Orden de Magnitud:** **$~10^4$ tokens** (Siete mil tokens aproximadamente).

---

#### 2. Imagen: 1 imagen de $1024 \times 1024$ píxeles
* **Supuestos explícitos:**
  * Los modelos de visión modernos (Vision Transformers / ViT) dividen las imágenes de alta resolución en cuadrículas de parches (*patches*) de menor tamaño (comúnmente de $512 \times 512$ o $256 \times 256$ píxeles).
  * Tomando como referencia el estándar de OpenAI (GPT-4V), la imagen primero se reduce a un cuadrado base si es muy grande, y luego se divide en parches de $512 \times 512$.
  * Una resolución de $1024 \times 1024$ genera exactamente una cuadrícula de $2 \times 2 = 4$ parches. Cada parche cuesta **170 tokens**, sumados a un costo base fijo de **85 tokens** por la imagen completa.
* **Cálculo:** $$\text{Tokens por parches} = 4 \text{ parches} \times 170 \text{ tokens} = 680 \text{ tokens}$$
  $$\text{Tokens totales} = 680 \text{ tokens} + 85 \text{ tokens (base)} = 765 \text{ tokens}$$
* **Orden de Magnitud:** **$~10^3$ tokens** (Menos de mil tokens).

---

#### 3. Audio: 2 minutos de audio crudo (16 kHz, 16-bit, Mono)
* **Supuestos explícitos:**
  * Los modelos nativos de audio (como Gemini o Whisper integrados) no procesan el archivo bit a bit directamente, sino que convierten el audio crudo en un espectrograma de frecuencias.
  * Por arquitectura estándar, el audio se fragmenta en ventanas temporales. El ratio típico de tokenización para modelos multimodales nativos oscila entre **20 y 50 tokens por cada segundo de audio** (independientemente de que la tasa de muestreo sea de 16 kHz). Tomaremos una media conservadora de **25 tokens/segundo**.
* **Cálculo:**
  $$\text{Total segundos} = 2 \text{ minutos} \times 60 \text{ segundos/minuto} = 120 \text{ segundos}$$
  $$\text{Total tokens} = 120 \text{ segundos} \times 25 \text{ tokens/segundo} = 3,000 \text{ tokens}$$
* **Orden de Magnitud:** **$~10^3$ tokens** (Tres mil tokens aproximadamente).

---

### 📈 Resumen del Presupuesto Total

| Modalidad | Cantidad Solicitada | Consumo Estimado (Tokens) | Orden de Magnitud |
| :--- | :--- | :--- | :--- |
| **Texto** | 10 páginas | ~7,000 | $10^4$ |
| **Imagen** | $1024 \times 1024$ px | ~765 | $10^3$ |
| **Audio** | 2 minutos | ~3,000 | $10^3$ |
| **TOTAL** | **Contexto Mixto** | **~10,765 tokens** | **$10^4$** |

## Actividad 2 — Estrategia de ventana

Describe cómo partirías un documento de 200k tokens para un modelo de 128k de ventana (resumen jerárquico, índice, etc.). Respuesta en español.


_Estrategia:_

...


### 🗺️ Estrategia: Resumen Jerárquico (Map-Reduce)

Esta estrategia es la ideal cuando necesitamos **comprender el documento en su totalidad**, realizar análisis globales o responder preguntas complejas que requieren conectar el principio con el final de un texto de 200k tokens usando una ventana limitada a 128k.

El proceso se divide en tres fases lógicas independientes:

```mermaid
graph TD
    A[📄 Documento Original: 200k tokens] -->|Dividir en Chunks| B[Chunk 1: 50k] & C[Chunk 2: 50k] & D[Chunk 3: 50k] & E[Chunk 4: 50k]
    B -->|Llamada LLM 1| F[📝 Resumen 1: 3k]
    C -->|Llamada LLM 2| G[📝 Resumen 2: 3k]
    D -->|Llamada LLM 3| H[📝 Resumen 3: 3k]
    E -->|Llamada LLM 4| I[📝 Resumen 4: 3k]
    F & G & H & I -->|Concatenar| J[📚 Contexto Condensado: ~12k tokens]
    J -->|Llamada LLM Final| K([🏁 Respuesta Global de Alta Calidad])

## Actividad 3 — API multimodal (stub)

Si tu curso usa un proveedor con visión, deja un **stub** que construya `messages` con una imagen (URL o path) + pregunta. Si no, comenta el formato esperado (`image_url`, etc.).


In [ ]:
# TODO: multimodal message structure (pseudo)
messages = [
    # {"role": "user", "content": [ ... ]}
]


# =====================================================================
# ACTIVIDAD 3: ESTRUCTURA DE MENSAJE MULTIMODAL (STUB / PSEUDOCÓDIGO)
# =====================================================================

# El estándar de la API requiere que el "content" ya no sea un simple string,
# sino una LISTA de diccionarios donde cada elemento especifica su 'type'.

messages = [
    {
        "role": "user",
        "content": [
            # 1. Componente de Texto: La pregunta o instrucción para el modelo
            {
                "type": "text",
                "text": "Dime cuántas palabras aproximadas hay en este documento y si ves alguna firma al final."
            },
            
            # 2. Componente de Imagen (Opción A: Por URL pública)
            {
                "type": "image_url",
                "image_url": {
                    "url": "https://ejemplo.com/imagenes/documento_escaneado.jpg",
                    "detail": "high"  # 'high' fuerza al modelo a mirar los detalles/patches
                }
            }
            
            # 3. Componente de Imagen (Opción B: Archivo Local en Base64)
            # Si la imagen está en tu computadora o en el Colab, se convierte a texto Base64:
            # {
            #     "type": "image_url",
            #     "image_url": {
            #         "url": f"data:image/jpeg;base64,{imagen_en_base64}"
            #     }
            # }
        ]
    }
]

# # Estructura de llamada típica (Comentada para evitar ejecuciones accidentales):
# print("Estructura multimodal (Stub) cargada correctamente en la variable 'messages'.")
# # respuesta = llm.invoke(messages)